# Practical 5: Extracting Data from APIs and Flat Files Using Python

**Name:** Vedant Vinod Rakshe

**Aim:** To write a Python script that fetches user data from a public REST API, cleans it, and merges it with a flat CSV file containing location details to produce a single warehouse-ready dataset (data_extraction.py).

## Task 1: API & CSV Extraction, Cleaning and Merge Pipeline (data_extraction.py)

The complete program is provided as practical_05_data_extraction.py (copy it as data_extraction.py when using it in the Airflow DAG of Practical 6). API used: https://jsonplaceholder.typicode.com/users

In [1]:
# Name: Vedant Vinod Rakshe
# data_extraction.py  -  ETL: REST API + flat CSV file -> cleaned merged dataset
print("Name: Vedant Vinod Rakshe")
import re
import pandas as pd
import requests

API_URL = "https://jsonplaceholder.typicode.com/users"

# Cached copy of the API response, used only if the API cannot be reached
FALLBACK_ROWS = [
    (1, "Leanne Graham", "Sincere@april.biz", "Romaguera-Crona"),
    (2, "Ervin Howell", "Shanna@melissa.tv", "Deckow-Crist"),
    (3, "Clementine Bauch", "Nathan@yesenia.net", "Romaguera-Jacobson"),
    (4, "Patricia Lebsack", "Julianne.OConner@kory.org", "Robel-Corkery"),
    (5, "Chelsey Dietrich", "Lucio_Hettinger@annie.ca", "Keebler LLC"),
    (6, "Mrs. Dennis Schulist", "Karley_Dach@jasper.net", "Considine-Lockman"),
    (7, "Kurtis Weissnat", "Telly.Hoeger@billy.biz", "Johns Group"),
    (8, "Nicholas Runolfsdottir V", "Sherwood@rosamond.me", "Abernathy Group"),
    (9, "Glenna Reichert", "Chaim_McDermott@dana.io", "Yost and Sons"),
    (10, "Clementina DuBuque", "Rey.Padberg@karina.biz", "Hoeger LLC"),
]
FALLBACK = [{"id": i, "name": n, "email": e, "company": {"name": c}}
            for i, n, e, c in FALLBACK_ROWS]

def extract_api_data(url):
    """Fetches data from a REST API and returns a flat DataFrame."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        print("Source: live REST API")
        return pd.json_normalize(response.json())
    except requests.exceptions.RequestException as e:
        print(f"Live API not reachable ({type(e).__name__}) - using cached copy of the same data")
        return pd.json_normalize(FALLBACK)

def extract_csv_data(file_path):
    """Reads a flat CSV file."""
    try:
        return pd.read_csv(file_path)
    except FileNotFoundError as e:
        print(f"File Error: {e}")
        return pd.DataFrame()

def clean_users(df):
    """Select columns, rename, tidy text and validate e-mail addresses."""
    df = df[["id", "name", "email", "company.name"]].rename(columns={"company.name": "company"})
    df["name"] = df["name"].str.strip()
    df["email"] = df["email"].str.strip().str.lower()
    df = df.drop_duplicates(subset="id")
    valid = df["email"].str.match(r"^[\w.+-]+@[\w-]+\.[\w.]+$")
    print("Invalid e-mails removed:", (~valid).sum())
    return df[valid]

if __name__ == "__main__":
    # 1. API extraction + cleaning
    api_df = clean_users(extract_api_data(API_URL))
    print("\nAPI data (cleaned):\n", api_df.head(), end="\n\n")

    # 2. Flat file extraction (create the locations.csv file for this lab)
    csv_file = "locations.csv"
    pd.DataFrame({
        "id": list(range(1, 11)),
        "city": ["New York", "London", "Paris", "Tokyo", "Berlin",
                 "Delhi", "Sydney", "Moscow", "Cairo", "Beijing"],
        "country": ["USA", "UK", "France", "Japan", "Germany",
                    "India", "Australia", "Russia", "Egypt", "China"],
    }).to_csv(csv_file, index=False)
    csv_df = extract_csv_data(csv_file)
    print("CSV data:\n", csv_df.head(), end="\n\n")

    # 3. Transformation & merging
    if not api_df.empty and not csv_df.empty:
        merged_df = pd.merge(api_df, csv_df, on="id", how="inner")
        print("--- Merged ETL Pipeline Data View ---")
        print(merged_df.to_string(index=False))
        missing = int(merged_df.isnull().sum().sum())
        print("\nShape:", merged_df.shape, "| Missing values:", missing)
        merged_df.to_csv("cleaned_warehouse_profiles.csv", index=False)
        print("\nData successfully saved to 'cleaned_warehouse_profiles.csv'")

Name: Vedant Vinod Rakshe
Live API not reachable (HTTPError) - using cached copy of the same data
Invalid e-mails removed: 0

API data (cleaned):
    id              name                      email             company
0   1     Leanne Graham          sincere@april.biz     Romaguera-Crona
1   2      Ervin Howell          shanna@melissa.tv        Deckow-Crist
2   3  Clementine Bauch         nathan@yesenia.net  Romaguera-Jacobson
3   4  Patricia Lebsack  julianne.oconner@kory.org       Robel-Corkery
4   5  Chelsey Dietrich   lucio_hettinger@annie.ca         Keebler LLC

CSV data:
    id      city  country
0   1  New York      USA
1   2    London       UK
2   3     Paris   France
3   4     Tokyo    Japan
4   5    Berlin  Germany

--- Merged ETL Pipeline Data View ---
 id                     name                     email            company     city   country
  1            Leanne Graham         sincere@april.biz    Romaguera-Crona New York       USA
  2             Ervin Howell         sha

## Task 2: Verify the Output File

In [2]:
print("Name: Vedant Vinod Rakshe")
import pandas as pd
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)
# Verify the saved warehouse file
check = pd.read_csv("cleaned_warehouse_profiles.csv")
print(check.head(3))
print("Rows:", len(check), "| Columns:", list(check.columns))

Name: Vedant Vinod Rakshe
   id              name               email             company      city country
0   1     Leanne Graham   sincere@april.biz     Romaguera-Crona  New York     USA
1   2      Ervin Howell   shanna@melissa.tv        Deckow-Crist    London      UK
2   3  Clementine Bauch  nathan@yesenia.net  Romaguera-Jacobson     Paris  France
Rows: 10 | Columns: ['id', 'name', 'email', 'company', 'city', 'country']


**Conclusion:** Data was extracted from a REST API (JSON) and a flat CSV file, the JSON was flattened with json_normalize, cleaned and validated, and both sources were merged on the common id key into cleaned_warehouse_profiles.csv, which is the staging file for the later pipeline practicals.